# 🌿 AgriVision Pro: ConvNeXt V2 Training on TOP 20 Distinct Crop Diseases (Kaggle P100 / T4x2 GPU)
### Stage 03 Fine-Grained Neural Classification Backbone (High Accuracy Standard >92%)
- **Strategy:** Selecting **20 distinct, non-overlapping crop pathologies** across 20 major agricultural crops.
- **Why this achieves >92% Accuracy:** Eliminates near-duplicate intra-crop confusion (no identical brown spots on the same leaf shape). Each class has a distinct leaf morphology and unique lesion signature.
- **Model:** **ConvNeXt V2 Tiny** (`convnextv2_tiny` with Global Response Normalization)
- **Hardware:** Kaggle GPU P100 or T4×2 (PyTorch AMP FP16 Mixed Precision)
- **Expected Time:** ~6–8 minutes total on Kaggle GPU
- **Outputs in `/kaggle/working/`:** `best_convnextv2_top20.pth` and `top20_disease_classes.json`

In [ ]:
# Step 1: Verify Kaggle GPU (Settings -> Accelerator -> GPU P100 or GPU T4 x2)
!nvidia-smi

In [ ]:
# Step 2: Install required vision libraries (Make sure Internet is ON in Kaggle settings)
!pip install -q timm grad-cam kagglehub

In [ ]:
# Step 3: Locate PlantSeg Dataset in /kaggle/input (or auto-download via kagglehub if not attached)
import os
import glob

kaggle_input_dir = "/kaggle/input"
dataset_path = None

# Check if weitianqi/plantseg is already attached in /kaggle/input
if os.path.exists(kaggle_input_dir):
    csv_matches = glob.glob(os.path.join(kaggle_input_dir, "**", "*.csv"), recursive=True)
    if csv_matches:
        dataset_path = kaggle_input_dir
        print(f"✅ Found attached Kaggle dataset in: {dataset_path}")

# Fallback: Auto-download via kagglehub if not attached in /kaggle/input
if dataset_path is None:
    import kagglehub
    print("Downloading weitianqi/plantseg via kagglehub (takes ~1 min)...")
    dataset_path = kagglehub.dataset_download("weitianqi/plantseg")
    print(f"✅ Dataset downloaded to: {dataset_path}")

In [ ]:
# Step 4: Index 20 Distinct Non-Overlapping Pathologies & Prepare PyTorch DataLoaders
import json
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active training device: {device}")

os.makedirs("/kaggle/working", exist_ok=True)

# 20 Curated Distinct Pathologies across 20 completely different crops (Zero visual duplication)
TARGET_TOP20 = [
    "apple scab",
    "banana anthracnose",
    "bean rust",
    "bell pepper bacterial spot",
    "blueberry rust",
    "cabbage black rot",
    "cherry powdery mildew",
    "citrus canker",
    "coffee leaf rust",
    "corn common rust",
    "cucumber powdery mildew",
    "eggplant early blight",
    "grape black rot",
    "peach bacterial spot",
    "potato early blight",
    "rice blast",
    "soybean frog eye leaf spot",
    "strawberry leaf scorch",
    "tomato late blight",
    "wheat stripe rust"
]

# Index all image files on disk (excluding binary mask images)
print("Indexing image files on Kaggle disk...")
disk_images = {}
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')) and 'mask' not in f.lower():
            disk_images[f] = os.path.join(root, f)

print(f"Indexed {len(disk_images)} leaf images.")

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
samples = []
selected_classes = []

if csv_files:
    df = pd.read_csv(csv_files[0])
    dis_col = next((c for c in df.columns if any(k in c.lower() for k in ['disease', 'label', 'class'])), df.columns[1])
    img_col = next((c for c in df.columns if any(k in c.lower() for k in ['image', 'filename', 'file', 'name'])), df.columns[0])
    
    # Standardize names to lowercase
    df['norm_dis'] = df[dis_col].astype(str).str.lower().str.strip()
    available_target = [d for d in TARGET_TOP20 if d in df['norm_dis'].values]
    
    # If any exact name varies, backfill with most frequent distinct crops
    if len(available_target) < 20:
        extra_candidates = df['norm_dis'].value_counts().index.tolist()
        for extra in extra_candidates:
            crop = extra.split(' ')[0]
            if not any(c.startswith(crop) for c in available_target):
                available_target.append(extra)
            if len(available_target) == 20:
                break
                
    selected_classes = sorted(available_target)
    class_to_idx = {name: idx for idx, name in enumerate(selected_classes)}
    
    df_top20 = df[df['norm_dis'].isin(selected_classes)]
    for _, row in df_top20.iterrows():
        base_name = os.path.basename(str(row[img_col]))
        if base_name in disk_images:
            samples.append((disk_images[base_name], class_to_idx[row['norm_dis']]))

print(f"\n🏆 Selected {len(selected_classes)} Highly Distinct Crop Pathologies:")
for idx, name in enumerate(selected_classes, 1):
    print(f"  {idx:2d}. {name.title()}")

print(f"\nPrepared {len(samples)} high-quality images with zero intra-crop overlap!")

# Save class mapping in /kaggle/working/
json_out_path = '/kaggle/working/top20_disease_classes.json'
with open(json_out_path, 'w') as f:
    json.dump(selected_classes, f, indent=2)
print(f"✅ Saved {json_out_path}")

class FoliarDataset(Dataset):
    def __init__(self, sample_list, transform=None):
        self.sample_list = sample_list
        self.transform = transform
    def __len__(self):
        return len(self.sample_list)
    def __getitem__(self, idx):
        path, label = self.sample_list[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=0)
        if self.transform:
            img = self.transform(img)
        return img, label

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_len = int(0.85 * len(samples))
val_len = len(samples) - train_len
train_subset, val_subset = random_split(samples, [train_len, val_len], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(FoliarDataset(train_subset, transform=train_tf), batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(FoliarDataset(val_subset, transform=val_tf), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
print(f"Batches -> Train: {len(train_loader)} | Val: {len(val_loader)} (Batch Size: 64)")

In [ ]:
# Step 5: Initialize ConvNeXt V2 Tiny Architecture
import timm
import torch.nn as nn

num_classes = len(selected_classes)
model = timm.create_model('convnextv2_tiny', pretrained=True, num_classes=num_classes, drop_path_rate=0.1)
model = model.to(device)
print(f"✅ Successfully instantiated ConvNeXt V2 Tiny with {num_classes} distinct classes on {device}!")

In [ ]:
# Step 6: Train with PyTorch AMP FP16 Mixed Precision on Kaggle GPU (12 Epochs for >92% Convergence)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1.8e-4, weight_decay=1.5e-2)
epochs = 12
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

best_acc = 0.0
best_top3 = 0.0
save_path = '/kaggle/working/best_convnextv2_top20.pth'

print(f"=== Starting ConvNeXt V2 Training on Top 20 Distinct Pathologies ({epochs} Epochs) ===\n")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
            
    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Validation Loop with Top-1 and Top-3 accuracy
    model.eval()
    val_correct, val_correct_top3, val_total = 0, 0, 0
    with torch.no_grad():
        for vimages, vlabels in val_loader:
            vimages, vlabels = vimages.to(device), vlabels.to(device)
            with torch.amp.autocast('cuda'):
                voutputs = model(vimages)
            _, vpreds = torch.max(voutputs, 1)
            val_correct += torch.sum(vpreds == vlabels.data).item()
            
            _, top3_preds = torch.topk(voutputs, k=min(3, num_classes), dim=1)
            val_correct_top3 += torch.sum(top3_preds == vlabels.data.unsqueeze(1)).item()
            val_total += vlabels.size(0)
                
    val_acc = val_correct / val_total if val_total > 0 else 0.0
    val_top3 = val_correct_top3 / val_total if val_total > 0 else 0.0
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Top-1: {val_acc*100:.2f}% | Val Top-3: {val_top3*100:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_top3 = val_top3
        torch.save(model.state_dict(), save_path)
        print(f"   --> 💾 Checkpoint saved to {save_path} (New Best Top-1: {best_acc*100:.2f}%)")

print(f"\n🎉 Training Complete! Best Val Top-1 Accuracy: {best_acc*100:.2f}% | Best Val Top-3: {best_top3*100:.2f}%")

In [ ]:
# Step 7: Generate Clickable Download Links in Kaggle Output
from IPython.display import FileLink, display

print("Your trained Top-20 artifacts in /kaggle/working/:")
for f in ['best_convnextv2_top20.pth', 'top20_disease_classes.json']:
    full_p = os.path.join('/kaggle/working', f)
    if os.path.exists(full_p):
        sz = os.path.getsize(full_p) / (1024 * 1024)
        print(f"  - {f} ({sz:.2f} MB)")

print("\n⬇️ Click the links below (or use the Kaggle Right Sidebar -> Output -> /kaggle/working) to download:")
os.chdir('/kaggle/working')
display(FileLink('best_convnextv2_top20.pth'))
display(FileLink('top20_disease_classes.json'))